# tadatorch: predicting transcriptional activation domains

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/idptools/tadatorch/blob/main/colab/tadatorch_colab.ipynb)

[tadatorch](https://github.com/idptools/tadatorch) predicts transcriptional activation domains (ADs) with the TADA network (Morffy, Van den Broeck et al. 2024, Nature 632, 166-173). It scores every 40 amino acid window of a sequence, and calls ADs from the windows that score at or above a threshold.

**To use this notebook:**

1. Run cell 1 to install tadatorch. This takes a minute or two, and only needs doing once per session.
2. Paste your sequence into cell 2, change any of the settings, and run it to make the plot.
3. If you want the plot as a PDF, run cell 3.

In [ ]:
# @title 1. Install tadatorch
# Packages are installed with uv. tadatorch currently needs sparrow from its
# nardini branch, which is installed in the same step so uv uses it for
# tadatorch's sparrow dependency.
!pip install --quiet uv
!uv pip install --system --quiet "idptools-sparrow @ git+https://github.com/idptools/sparrow.git@nardini" "tadatorch @ git+https://github.com/idptools/tadatorch.git"

**The plot** has two panels that share the residue axis:

* **Bottom:** the per-residue TAD score (black) and the disorder score predicted by [metapredict](https://github.com/idptools/metapredict) V3 (red), both from 0 to 1. Each residue's TAD score is the highest score of the windows it sits at the center of, so the line does not reach the first or last 19 residues, which no window is centered on.
* **Top:** the ADs called with the settings below, drawn as blue rectangles along the sequence.

With the default `whole_window` method, ADs are at least 40 residues long and extend past where the TAD score crosses the threshold. `central_residues` gives tighter ADs that follow the TAD score line. See the [tadatorch README](https://github.com/idptools/tadatorch#how-activation-domains-are-annotated) for details of both.

In [ ]:
# @title 2. Predict and plot activation domains
# tadatorch makes figures without pyplot, so this turns on showing them in the notebook
%matplotlib inline

import logging

from IPython.display import display

from tadatorch.backend.domains import find_activation_domains
from tadatorch.plotting import plot_sequence_profile
from tadatorch.tada import predict_window_scores

# Colab does not have Arial, so the plot falls back to matplotlib's default
# font. This hides the warning matplotlib gives about that for every label.
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)

# @markdown Paste in an amino acid sequence of at least 40 residues (spaces and line breaks are removed, and letters are made uppercase). The example is yeast Gcn4 (UniProt P03069).
sequence = "MSEYQPSLFALNPMGFSPLDGSKSTNENVSASTSTAKPMVGQLIFDKFIKTEEDPIIKQDTPSNLDFDFALPQTATAPDAKTVLPIPELDDAVVESFFSSSTDSTPMFEYENLEDNSKEWTSLFDNDIPVTTDDVSLADKAIESTEEVSLVPSNLEVSTTSFLPTPVLEDAKLTQTRKVKKPNSVVKKSHHVGKDDESRLDHLGVVAYNRKQRSIPLSPIVPESSDPAALKRARNTEAARRSRARKLQRMKQLEDKVEELLSKNYHLENEVARLKKLVGER"  # @param {type:"string"}
name = "Gcn4"  # @param {type:"string"}
# @markdown Windows with a TAD score at or above the threshold (0 to 1) are treated as ADs. 0.5 is the threshold used in the TADA paper.
threshold = 0.5  # @param {type:"slider", min:0.0, max:1.0, step:0.01}
# @markdown How ADs are called: `whole_window` marks every residue of a window at or above the threshold, and `central_residues` only marks its two central residues.
method = "whole_window"  # @param ["whole_window", "central_residues"]
# @markdown Only used by `central_residues`: gaps between ADs shorter than `gap_size` residues are joined, and ADs shorter than `min_ad_size` residues are removed.
gap_size = 6  # @param {type:"integer"}
min_ad_size = 6  # @param {type:"integer"}

# remove any spaces or line breaks pasted in with the sequence
sequence = "".join(sequence.split()).upper()

window_scores = predict_window_scores(sequence)
domains = find_activation_domains(
    sequence,
    window_scores,
    threshold=threshold,
    method=method,
    gap_size=gap_size,
    min_ad_size=min_ad_size,
)

figure = plot_sequence_profile(name, sequence, window_scores, domains)
display(figure)

print(
    f"Found {len(domains)} AD(s) in {name} ({len(sequence)} residues) using a threshold of {threshold} and the {method} method."
)
for domain in domains:
    print(
        f"  Residues {domain.start}-{domain.end} ({len(domain.sequence)} residues), "
        f"max window score {domain.max_score:.3f}: {domain.sequence}"
    )

In [ ]:
# @title 3. (Optional) Download the plot as a PDF
# open_profile_pdf saves text as TrueType fonts, so it can be edited in Illustrator
from google.colab import files

from tadatorch.plotting import open_profile_pdf

pdf_path = "tadatorch_plot.pdf"
with open_profile_pdf(pdf_path) as pdf:
    pdf.savefig(figure)

files.download(pdf_path)